<a href="https://colab.research.google.com/github/bvsvntv/ai-ml-learning/blob/main/notebooks/lang-graph/fault_tolerance_using_checkpoints.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time
from pprint import pprint

In [2]:
# state
class CrashState(TypedDict):
  input: str
  step1: bool
  step2: bool

In [3]:
def step1(state: CrashState) -> CrashState:
  print("step 1 executed")
  return {"step1": True, "input": state["input"]}

In [11]:
def step2(state: CrashState) -> CrashState:
  print("step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
  time.sleep(30)
  print("step 2")

In [5]:
def step3(state: CrashState):
  print("step 3 executed")
  return {"step3": True}

In [12]:
graph = StateGraph(CrashState)

# add nodes
graph.add_node("step_1", step1)
graph.add_node("step_2", step2)
graph.add_node("step_3", step3)

graph.add_edge(START, "step_1")
graph.add_edge("step_1", "step_2")
graph.add_edge("step_2", "step_3")
graph.add_edge("step_3", END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [13]:
try:
  print("running graph, manually interrupt during step 2")
  workflow.invoke({"input": "start"}, config={"configurable": {"thread_id": "thread_1"}})
except KeyboardInterrupt:
  print("kernel manually interrupted (crash simulated)")

running graph, manually interrupt during step 2
step 1 executed
step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
kernel manually interrupted (crash simulated)


In [14]:
workflow.get_state(config={"configurable": {"thread_id": "thread_1"}})

StateSnapshot(values={'input': 'start', 'step1': True}, next=('step_2',), config={'configurable': {'thread_id': 'thread_1', 'checkpoint_ns': '', 'checkpoint_id': '1f1de87e-92c8-64c7-8001-180862dd53d3'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-11-12T10:57:53.915600+00:00', parent_config={'configurable': {'thread_id': 'thread_1', 'checkpoint_ns': '', 'checkpoint_id': '1f1de87e-92c4-646d-8000-07aeaddb5895'}}, tasks=(PregelTask(id='e7c2d8db-bdfa-e618-4b9d-48e4b3a433c8', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [15]:
# re-run to show fault tolerance
print("re-running graph to demonstrate fault tolerance...")
final_state = workflow.invoke(None, config={"configurable": {"thread_id": "thread_1"}})
print("final state: ", final_state)

re-running graph to demonstrate fault tolerance...
step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
step 2
step 3 executed
final state:  {'input': 'start', 'step1': True}


In [18]:
pprint(list(workflow.get_state_history(config={"configurable": {"thread_id": "thread_1"}})))

[StateSnapshot(values={'input': 'start', 'step1': True}, next=(), config={'configurable': {'thread_id': 'thread_1', 'checkpoint_ns': '', 'checkpoint_id': '1f1de87f-fc5e-6528-8003-27b23a408471'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-11-12T10:58:31.830641+00:00', parent_config={'configurable': {'thread_id': 'thread_1', 'checkpoint_ns': '', 'checkpoint_id': '1f1de87f-fc5b-6867-8002-17c38c54aa1c'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': True}, next=('step_3',), config={'configurable': {'thread_id': 'thread_1', 'checkpoint_ns': '', 'checkpoint_id': '1f1de87f-fc5b-6867-8002-17c38c54aa1c'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-11-12T10:58:31.829487+00:00', parent_config={'configurable': {'thread_id': 'thread_1', 'checkpoint_ns': '', 'checkpoint_id': '1f1de87e-92c8-64c7-8001-180862dd53d3'}}, tasks=(PregelTask(id='18d831eb-f107-4750-24b5-90c097ca9509', name='step_3', path=('__pregel_pul